<a href="https://colab.research.google.com/github/Shorovpaul/Data_Testing/blob/main/Adult_census_income.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy scikit-learn imbalanced-learn shap xgboost

In [ ]:
import pandas as pd

data = pd.read_csv("adult_census_income.csv")

print(data.shape)
# Step 1: Rename the column
data.rename(columns={'income': 'Class'}, inplace=True)

# Step 2: Verify it worked
print(data.columns.tolist())

# Step 3: Now use it
display(data['Class'].value_counts())

(32561, 15)
['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'Class']


,count
Class,
<=50K,24720
>50K,7841


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

X = data.drop("Class", axis=1)
y = data["Class"]

# Encode target variable
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Identify categorical columns in the training set
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# Combine X_train and X_test to apply get_dummies consistently
# This ensures both dataframes have the same columns after encoding.
X_combined = pd.concat([X_train, X_test], ignore_index=True)
X_combined_encoded = pd.get_dummies(X_combined, columns=categorical_cols, drop_first=True)

# Split back into X_train and X_test
X_train = X_combined_encoded.iloc[:len(X_train)]
X_test = X_combined_encoded.iloc[len(X_train):]

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd # Import pandas to use Series.value_counts()

smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(pd.Series(y_train_bal).value_counts())

1    19775
0    19775
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_bal = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

models = {
    "LR": LogisticRegression(max_iter=1000),
    "RF": RandomForestClassifier(n_estimators=200),
    "GB": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "XGB": xgb.XGBClassifier(eval_metric="logloss")
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

In [ ]:
 from sklearn.metrics import average_precision_score

model_scores = {}

for name, model in trained_models.items():
    probs = model.predict_proba(X_test_scaled)[:,1]
    score = average_precision_score(y_test, probs)
    model_scores[name] = score

print(model_scores)

{'LR': np.float64(0.7385634607703038), 'RF': np.float64(0.738610250907076), 'GB': np.float64(0.7810381029013694), 'KNN': np.float64(0.5815893406620771), 'SVM': np.float64(0.7243311031639714), 'XGB': np.float64(0.8046508572465916)}


In [39]:
selected_models = {
    name:model for name,model in trained_models.items()
    if model_scores[name] > 0.70
}

In [ ]:
import shap
import numpy as np
# Import all model classes used in selected_models for isinstance checks
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb

explanations = {}

# Use a very small subset of the training data as background for KernelExplainer
# KernelExplainer is computationally intensive, so a smaller background can speed things up.
background_data_for_kernel = X_train_bal[:50]

for name, model in selected_models.items():
    print(f"Explaining model: {name}") # Debugging print

    if isinstance(model, (RandomForestClassifier, GradientBoostingClassifier, xgb.XGBClassifier)):
        # For tree-based models, use TreeExplainer.
        # The second argument to TreeExplainer is the background data.
        explainer = shap.TreeExplainer(model, X_train_bal[:1000])
        # Calculate SHAP values for the first 1000 samples of the training data
        # Setting check_additivity=False addresses the ExplainerError encountered previously.
        shap_values_obj = explainer(X_train_bal[:1000], check_additivity=False)
        # For Explanation object, SHAP values are in .values
        importance = np.abs(shap_values_obj.values).mean(axis=0)
    elif isinstance(model, (LogisticRegression, SVC)):
        # For other models like Logistic Regression and SVM, use KernelExplainer.
        # KernelExplainer requires a function that returns probabilities and a background dataset.
        explainer = shap.KernelExplainer(model.predict_proba, background_data_for_kernel)
        # Calculate SHAP values for the first 1000 samples of the training data
        shap_values_list = explainer.shap_values(X_train_bal[:1000])
        # For binary classification with KernelExplainer, shap_values_list is a list of two arrays.
        # We are interested in the SHAP values for the positive class (index 1).
        importance = np.abs(shap_values_list[1]).mean(axis=0)
    else:
        print(f"Warning: No specific explainer implemented for model type {type(model).__name__}. Skipping.")
        continue # Skip to the next model

    explanations[name] = importance

print("Explanations generated successfully.")

Explaining model: LR


  0%|          | 0/1000 [00:00<?, ?it/s]

Explaining model: RF


100%|===================| 1999/2000 [08:34<00:00]       

Explaining model: GB
Explaining model: SVM


  0%|          | 0/1000 [00:00<?, ?it/s]